In [7]:
"""
Coral Age Model Correction via Monotonic Time Warping
======================================================
Matches coral sample ages to a climate-based d18O proxy by optimizing
each sample's age individually, while preserving the original sample order.

Inputs:
    - interp_20251220.xlsx  : coral data with 'age_absolute' and 'd18o' columns
    - climate_df.csv        : climate proxy with 'age_absolute' and 'synthetic_d18o' columns

Output:
    - coral_age_warped.xlsx : original data + new 'age_warped' column
"""

import numpy as np
import pandas as pd
from scipy.interpolate import interp1d, UnivariateSpline
from scipy.optimize import minimize
from scipy.stats import pearsonr

# =============================================================================
# 1. Load data
# =============================================================================
df_coral = pd.read_excel("../../../2025-12-20_AgeModel-NewXrayPatch/age_model/20260218/interp_20260218.xlsx")
df_clim = pd.read_csv("../../../2026-01-15_Climate-Model/analysis/climate_df.csv")

# =============================================================================
# 2. Subset coral to the overlap period with climate data
# =============================================================================
clim_age = df_clim["age_absolute"].values
clim_d18o = df_clim["synthetic_d18o"].values

coral_sub = (
    df_coral[
        (df_coral["age_absolute"] >= clim_age.min()) &
        (df_coral["age_absolute"] <= clim_age.max())
    ]
    .copy()
    .reset_index(drop=True)
)

coral_d18o = coral_sub["d18o"].values
orig_ages = coral_sub["age_absolute"].values

# =============================================================================
# 3. Build a fast climate interpolator
# =============================================================================
f_clim = interp1d(clim_age, clim_d18o, bounds_error=False, fill_value=np.nan)

# =============================================================================
# 4. Parametrize ages to guarantee monotonicity
#
#   Instead of optimizing ages directly (which could violate ordering),
#   we parametrize as:
#       ages[0]   = orig_ages[0] + params[0]          (free start offset)
#       ages[i+1] = ages[i]      + exp(params[i+1])   (always positive spacing)
#
#   This means params can be any real numbers, and the resulting ages
#   are always strictly increasing — no ordering constraint needed.
# =============================================================================
def ages_from_params(params, orig_ages):
    start = orig_ages[0] + params[0]
    spacings = np.exp(params[1:])           # exp() ensures positive spacing
    ages = np.concatenate([[start], start + np.cumsum(spacings)])
    return ages


def objective(params, orig_ages, coral_d18o, f_clim):
    ages = ages_from_params(params, orig_ages)

    # Hard boundary: keep ages within the climate record
    if ages[-1] > f_clim.x.max() or ages[0] < f_clim.x.min():
        return 1e6

    clim_vals = f_clim(ages)
    if np.any(np.isnan(clim_vals)):
        return 1e6

    r, _ = pearsonr(clim_vals, coral_d18o)
    return -r   # minimize negative correlation = maximize correlation


# =============================================================================
# 5. Set up initial parameters and bounds
# =============================================================================

# Fix near-duplicate ages (numerical safety)
orig_ages_safe = orig_ages.copy()
for i in range(1, len(orig_ages_safe)):
    if orig_ages_safe[i] <= orig_ages_safe[i - 1]:
        orig_ages_safe[i] = orig_ages_safe[i - 1] + 1e-5

orig_spacings = np.diff(orig_ages_safe)
orig_spacings = np.where(orig_spacings <= 0, 1e-4, orig_spacings)

# Initial guess: ~3 year global offset (from prior cross-correlation step),
# original spacings for the rest
init_log_spacings = np.log(orig_spacings)
init_params = np.concatenate([[3.1], init_log_spacings])

# Bounds:
#   - Start offset: -0.5 to +6 years
#   - Each interval: up to 2x compression or 2x expansion of original spacing
bounds = [(-0.5, 6.0)] + [
    (log_s - np.log(2), log_s + np.log(2))
    for log_s in init_log_spacings
]

# =============================================================================
# 6. Optimize
# =============================================================================
print(f"Initial correlation: {-objective(init_params, orig_ages_safe, coral_d18o, f_clim):.4f}")

result = minimize(
    objective,
    init_params,
    args=(orig_ages_safe, coral_d18o, f_clim),
    method="L-BFGS-B",
    bounds=bounds,
    options={"maxiter": 10000, "ftol": 1e-12, "maxfun": 200000},
)

final_ages = ages_from_params(result.x, orig_ages_safe)
age_diff = final_ages - orig_ages_safe

print(f"Optimized correlation: {-result.fun:.4f}")
print(f"Age corrections — mean: {age_diff.mean():.3f} yr, "
      f"std: {age_diff.std():.3f} yr, "
      f"range: [{age_diff.min():.3f}, {age_diff.max():.3f}]")

# =============================================================================
# 7. Extrapolate corrections to the full dataset (outside overlap window)
#    using a smooth spline fitted to the in-overlap corrections
# =============================================================================
sort_idx = np.argsort(orig_ages_safe)
corr_spline = UnivariateSpline(
    orig_ages_safe[sort_idx],
    age_diff[sort_idx],
    s=5,    # smoothing factor — increase to smooth more
    k=3,    # cubic spline
    ext=3,  # extrapolate as boundary value beyond edges
)

df_coral["age_warped"] = df_coral["age_absolute"] + corr_spline(df_coral["age_absolute"])

# Sanity check: should be monotonically increasing
sorted_warped = df_coral.sort_values("age_absolute")["age_warped"].values
assert np.all(np.diff(sorted_warped) >= 0), "Monotonicity violated after extrapolation!"

# =============================================================================
# 8. Save output
# =============================================================================
df_coral.to_excel("coral_age_warped.xlsx", index=False)
print("Saved: coral_age_warped.xlsx")

Initial correlation: 0.2476
Optimized correlation: 0.6961
Age corrections — mean: 2.923 yr, std: 0.230 yr, range: [2.115, 3.480]
Saved: coral_age_warped.xlsx
